---
title: "动态规划 (DP)——状态机动态规划"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
from typing import List

## 📝 题目 300：最长递增子序列

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `nums` ，找到其中最长严格递增子序列的长度。

**子序列** 是由数组派生而来的序列，删除（或不删除）数组中的元素而不改变其余元素的顺序。例如，`[3,6,2,7]` 是数组 `[0,3,1,6,2,2,7]` 的子序列。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [10,9,2,5,3,7,101,18]`
    * **输出**：`4`
    * **解释**：最长递增子序列是 `[2,3,7,101]`，因此长度为 4 。

* **示例 2**：
    * **输入**：`nums = [0,1,0,3,2,3]`
    * **输出**：`4`

* **示例 3**：
    * **输入**：`nums = [7,7,7,7,7,7,7]`
    * **输出**：`1`
:::

In [2]:
class Solution300:
    def lengthOfLIS(self, nums: List[int]) -> int:
        if not nums:
            return 0
        dp = [1] * len(nums)  # 长度保底是 1
        for i in range(len(nums)):
            for j in range(i):
                if nums[i] > nums[j]:
                    dp[i] = max(dp[i], dp[j] + 1)  # 状态转移：取自己目前的长度，和接在 j 后面的长度的最大值
        return max(dp)

In [3]:
#| code-fold: true
def test_300(func):
    test_cases = [
        {"desc": "示例 1 (常规乱序)", "nums": [10, 9, 2, 5, 3, 7, 101, 18], "expected": 4},
        {"desc": "示例 2 (含重复数字)", "nums": [0, 1, 0, 3, 2, 3], "expected": 4},
        {"desc": "示例 3 (全相等触发保底)", "nums": [7, 7, 7, 7, 7, 7, 7], "expected": 1},
        {"desc": "完全递减 (触发保底)", "nums": [5, 4, 3, 2, 1], "expected": 1},
        {"desc": "完全递增", "nums": [1, 2, 3, 4, 5], "expected": 5},
        {"desc": "只有一个元素", "nums": [42], "expected": 1},
        {"desc": "负数与跳跃", "nums": [4, -1, 2, -3, 5, 8], "expected": 4} # -1 -> 2 -> 5 -> 8
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_300(Solution300().lengthOfLIS)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 4    | 4    | 示例 1 (常规乱序)
2    | ✅ PASS | 4    | 4    | 示例 2 (含重复数字)
3    | ✅ PASS | 1    | 1    | 示例 3 (全相等触发保底)
4    | ✅ PASS | 1    | 1    | 完全递减 (触发保底)
5    | ✅ PASS | 5    | 5    | 完全递增
6    | ✅ PASS | 1    | 1    | 只有一个元素
7    | ✅ PASS | 4    | 4    | 负数与跳跃
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

对于子序列问题，我们通常**强制要求状态以当前元素结尾**，这样才能方便地和前面的元素进行大小比较。

1. **确定 dp 数组以及下标的含义**：
   - 定义一维数组 `dp`。
   - `dp[i]` 表示：**以 `nums[i]` 这个数结尾**的，最长递增子序列的长度。

2. **确定递推公式（状态转移方程）**：
   - 假设我们现在要求 `dp[i]`，也就是以 `nums[i]` 结尾的最长长度。
   - 我们该怎么做？很简单，**回头看！**
   - 我们回头遍历从 `0` 到 `i-1` 的每一个元素（设下标为 `j`）。
   - 如果发现 `nums[i] > nums[j]`，说明 `nums[i]` 可以名正言顺地接在 `nums[j]` 的屁股后面，形成一个更长的递增子序列！
   - 接上去之后的长度是多少？就是 `dp[j] + 1`。
   - 因为前面可能有好几个比 `nums[i]` 小的数，我们要挑一个能拼出最长结果的。
   - **方程：`if nums[i] > nums[j]: dp[i] = max(dp[i], dp[j] + 1)`**

3. **dp 数组如何初始化**：
   - 每一个数字本身，都可以独立成为一个长度为 1 的递增子序列。
   - 所以，整个 `dp` 数组一开始全部初始化为 `1`。

4. **确定遍历顺序**：
   - 外层循环 `i` 从 `1` 遍历到 `len(nums) - 1`（当前要处理的数字）。
   - 内层循环 `j` 从 `0` 遍历到 `i - 1`（回头看前面的所有数字）。
   - 正序遍历。

5. **最终结果（⚠️ 极易踩坑的盲点）**：
   - 最终的最长递增子序列，**不一定**是以数组最后一个元素结尾的！它可能在数组的任何位置达到最长。
   - 所以，最后的结果不是返回 `dp[-1]`，而是要**返回整个 `dp` 数组中的最大值 `max(dp)`**。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N^2)$。两层 `for` 循环。
* **空间复杂度**：$O(N)$。需要一个长度为 `N` 的一维 `dp` 数组。
*(注：这道题还有一个进阶的贪心 + 二分查找解法，时间复杂度可以降到 $O(N \log N)$，但在目前的 DP 阶段，我们先专注把 $O(N^2)$ 的转移逻辑写透彻。)*
:::

## 📝 题目 1143：最长公共子序列

::: {.callout-caution}
### 📖 题目描述
给定两个字符串 `text1` 和 `text2`，返回这两个字符串的最长 **公共子序列** 的长度。如果不存在 **公共子序列** ，返回 `0` 。

一个字符串的 **子序列** 是指这样一个新的字符串：它是由原字符串在不改变字符的相对顺序的情况下删除某些字符（也可以不删除任何字符）后组成的新字符串。
例如，`"ace"` 是 `"abcde"` 的子序列，但 `"aec"` 不是 `"abcde"` 的子序列。

两个字符串的 **公共子序列** 是这两个字符串所共同拥有的子序列。

**输入输出示例**：

* **示例 1**：
    * **输入**：`text1 = "abcde", text2 = "ace"`
    * **输出**：`3`
    * **解释**：最长公共子序列是 `"ace"` ，它的长度为 3 。

* **示例 2**：
    * **输入**：`text1 = "abc", text2 = "abc"`
    * **输出**：`3`
    * **解释**：最长公共子序列是 `"abc"` ，它的长度为 3 。

* **示例 3**：
    * **输入**：`text1 = "abc", text2 = "def"`
    * **输出**：`0`
    * **解释**：两个字符串没有公共子序列，返回 0 。
:::

In [4]:
class Solution1143:
    def longestCommonSubsequence(self, text1: str, text2: str) -> int:
        dp = [[0] * (len(text2) + 1) for _ in range(len(text1) + 1)]
        for i in range(1, len(text1) + 1):
            for j in range(1, len(text2) + 1):
                if text1[i - 1] == text2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[-1][-1]

In [5]:
#| code-fold: true
def test_1143(func):
    test_cases = [
        {"desc": "示例 1 (常规)", "text1": "abcde", "text2": "ace", "expected": 3},
        {"desc": "示例 2 (完全相同)", "text1": "abc", "text2": "abc", "expected": 3},
        {"desc": "示例 3 (毫无瓜葛)", "text1": "abc", "text2": "def", "expected": 0},
        {"desc": "text1 比 text2 长很多 (越界测试)", "text1": "abcdefghij", "text2": "xyz", "expected": 0},
        {"desc": "text2 比 text1 长很多 (越界测试)", "text1": "a", "text2": "aaaaaa", "expected": 1},
        {"desc": "交错重叠", "text1": "oxcpqrsvwf", "text2": "shmtulvdsp", "expected": 2}, # 例如 "s", "p"
        {"desc": "包含空格", "text1": "hello world", "text2": "ho wd", "expected": 5}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["text1"], tc["text2"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1143(Solution1143().longestCommonSubsequence)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 3    | 3    | 示例 1 (常规)
2    | ✅ PASS | 3    | 3    | 示例 2 (完全相同)
3    | ✅ PASS | 0    | 0    | 示例 3 (毫无瓜葛)
4    | ✅ PASS | 0    | 0    | text1 比 text2 长很多 (越界测试)
5    | ✅ PASS | 1    | 1    | text2 比 text1 长很多 (越界测试)
6    | ✅ PASS | 2    | 2    | 交错重叠
7    | ✅ PASS | 5    | 5    | 包含空格
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

既然是两个字符串的比对，一维数组已经装不下这么多状态了。我们需要祭出**二维网格（矩阵）**！



想象我们把 `text1` 放在矩阵的行，`text2` 放在矩阵的列。

1. **确定 dp 数组以及下标的含义（🔥 究极防坑技巧：长度加一）**：
   - 定义一个二维数组 `dp`，大小为 `(len(text1) + 1) \times (len(text2) + 1)`。
   - 为什么要加一？为了留出第一行和第一列作为“空字符串”的基准线，这样可以完美避开各种数组越界判断！
   - `dp[i][j]` 表示：`text1` 的前 `i` 个字符（即 `text1[0:i-1]`）和 `text2` 的前 `j` 个字符（即 `text2[0:j-1]`）的最长公共子序列长度。

2. **确定递推公式（状态转移方程）**：
   - 我们在比对 `text1` 的第 `i` 个字符（`text1[i-1]`）和 `text2` 的第 `j` 个字符（`text2[j-1]`）时，只有两种情况：
   - **情况 1：这俩字符一模一样！**
     - 太棒了！说明我们找到了一个公共字符。这个字符可以接在它们俩前面字符串的公共子序列后面。
     - 那么当前的长度就是：**左上角**的长度加 1。
     - **方程：`dp[i][j] = dp[i-1][j-1] + 1`**
   - **情况 2：这俩字符不一样！**
     - 说明这俩字符不可能同时出现在公共子序列里。我们要么放弃 `text1` 的当前字符，要么放弃 `text2` 的当前字符。
     - 怎么选？当然是看哪种情况留下的子序列更长！
     - 即看**正上方** (`dp[i-1][j]`) 和 **正左方** (`dp[i][j-1]`) 哪个大。
     - **方程：`dp[i][j] = max(dp[i-1][j], dp[i][j-1])`**

3. **dp 数组如何初始化**：
   - 第一行：`text1` 是空字符串，与任何 `text2` 的公共子序列长度都是 0。
   - 第一列：`text2` 是空字符串，与任何 `text1` 的公共子序列长度也是 0。
   - 所以，整个二维数组全部初始化为 `0` 即可！

4. **确定遍历顺序**：
   - 从上到下，从左到右，正序遍历两个字符串。外层循环 `i` 从 `1` 到 `len(text1)`，内层循环 `j` 从 `1` 到 `len(text2)`。

5. **最终结果**：
   - 右下角的那个格子 `dp[-1][-1]` 就是我们要的终极答案！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N)$。其中 $M$ 和 $N$ 分别是两个字符串的长度。我们需要填满一个 $M \times N$ 的二维矩阵。
* **空间复杂度**：$O(M \times N)$。用于构建二维 dp 数组。
:::

## 📝 题目 72：编辑距离

::: {.callout-caution}
### 📖 题目描述
给你两个单词 `word1` 和 `word2`， 请返回将 `word1` 转换成 `word2` 所使用的最少操作数  。

你可以对一个单词进行如下三种操作：
1. **插入**一个字符
2. **删除**一个字符
3. **替换**一个字符

**输入输出示例**：

* **示例 1**：
    * **输入**：`word1 = "horse", word2 = "ros"`
    * **输出**：`3`
    * **解释**：
      horse -> rorse (将 'h' 替换为 'r')
      rorse -> rose (删除 'r')
      rose -> ros (删除 'e')

* **示例 2**：
    * **输入**：`word1 = "intention", word2 = "execution"`
    * **输出**：`5`
    * **解释**：
      intention -> inention (删除 't')
      inention -> enention (将 'i' 替换为 'e')
      enention -> exention (将 'n' 替换为 'x')
      exention -> exection (将 'n' 替换为 'c')
      exection -> execution (插入 'u')
:::

In [8]:
class Solution72:
    def minDistance(self, word1: str, word2: str) -> int:
        dp = [[0] * (len(word2) + 1) for _ in range(len(word1) + 1)]
        for i in range(len(word1) + 1):  # 空字符串变为 word2 只能插入
            dp[i][0] = i
        for j in range(len(word2) + 1):  # word1 变为空字符串只能删除
            dp[0][j] = j
        for i in range(1, len(word1) + 1):
            for j in range(1, len(word2) + 1):
                if word1[i - 1] == word2[j - 1]:  # 最后一个字符相等
                    dp[i][j] = dp[i - 1][j - 1]
                else:  # 最后一个字符不相等时，替换：dp[i - 1][j - 1]，删除：dp[i - 1][j]，插入：dp[i][j - 1]
                    dp[i][j] = min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1]) + 1
        return dp[-1][-1]

In [7]:
#| code-fold: true
def test_72(func):
    test_cases = [
        {"desc": "示例 1 (综合操作)", "word1": "horse", "word2": "ros", "expected": 3},
        {"desc": "示例 2 (长单词比对)", "word1": "intention", "word2": "execution", "expected": 5},
        {"desc": "完全一样", "word1": "abc", "word2": "abc", "expected": 0},
        {"desc": "一个为空 (纯删除)", "word1": "hello", "word2": "", "expected": 5},
        {"desc": "另一个为空 (纯插入)", "word1": "", "word2": "world", "expected": 5},
        {"desc": "完全不交叉 (全替换)", "word1": "cat", "word2": "dog", "expected": 3},
        {"desc": "交错修改", "word1": "zoologicoarchaeologist", "word2": "zoogeologist", "expected": 10}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["word1"], tc["word2"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_72(Solution72().minDistance)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 3    | 3    | 示例 1 (综合操作)
2    | ✅ PASS | 5    | 5    | 示例 2 (长单词比对)
3    | ✅ PASS | 0    | 0    | 完全一样
4    | ✅ PASS | 5    | 5    | 一个为空 (纯删除)
5    | ✅ PASS | 5    | 5    | 另一个为空 (纯插入)
6    | ✅ PASS | 3    | 3    | 完全不交叉 (全替换)
7    | ✅ PASS | 10   | 10   | 交错修改
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解



我们要把 `word1` 变成 `word2`，依然使用**二维“长加一”网格**！

1. **确定 dp 数组以及下标的含义**：
   - 定义二维数组 `dp`，大小为 `(len(word1) + 1) \times (len(word2) + 1)`。
   - `dp[i][j]` 表示：将 `word1` 的前 `i` 个字符转换成 `word2` 的前 `j` 个字符，所需要的最少操作步数。

2. **dp 数组如何初始化（🔥 本题最大陷阱，不再全是 0 了！）**：
   - **第一行 (`i = 0`)**：`word1` 是空字符串。你要把空字符串变成长度为 `j` 的 `word2`，只能疯狂**插入**。所以 `dp[0][j] = j`。
   - **第一列 (`j = 0`)**：`word2` 是空字符串。你要把长度为 `i` 的 `word1` 变成空，只能疯狂**删除**。所以 `dp[i][0] = i`。
   - *(这就像是你在一张白纸上写字，或者把写好的字全擦掉，步数就是字的长度！)*

3. **确定递推公式（状态转移方程）**：
   - 我们在比对 `word1[i-1]` 和 `word2[j-1]` 时，依然是两种情况：
   - **情况 1：这俩字符一模一样！**
     - 太棒了！不需要任何额外操作。直接继承它俩前面字符串的编辑距离。
     - **方程：`dp[i][j] = dp[i-1][j-1]`** *(注意这里不加 1，因为没做任何操作)*
   - **情况 2：这俩字符不一样！（终极博弈开始）**
     - 我们有三种魔法，看看哪种魔法代价最小：
       - **替换 (Replace)**：把 `word1[i-1]` 变成 `word2[j-1]`。操作数等于左上角的步数 + 1。也就是 `dp[i-1][j-1] + 1`。
       - **删除 (Delete)**：把 `word1[i-1]` 删掉，指望 `word1` 前面的部分能和 `word2` 匹配。操作数等于正上方的步数 + 1。也就是 `dp[i-1][j] + 1`。
       - **插入 (Insert)**：在 `word1` 屁股后面插一个和 `word2[j-1]` 一样的字符，指望 `word1` 本身能和 `word2` 前面的部分匹配。操作数等于正左方的步数 + 1。也就是 `dp[i][j-1] + 1`。
     - **方程：`dp[i][j] = min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1]) + 1`**

4. **最终结果**：
   - 依然是右下角的那个格子 `dp[-1][-1]`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N)$。填满整个二维矩阵。
* **空间复杂度**：$O(M \times N)$。二维 `dp` 数组的开销。
:::